# 03 — Which family of model, and what learning was worth

The question is not « which model is best ». It is whether the choice of family changes the
cost of a decision, and by how much, against two baselines that decide without looking at
the applicant at all.

Everything below is read from `reports/model_comparison.csv`, written by
`scripts/compare_models.py`. The script cross-validates each family on the same folds of the
same frame, with the same cost ratio, and writes the table down. Nothing here recomputes a
number, and nothing here is typed by hand.

**The comparison runs on the holdout**, 30 751 applicants: the Home Credit data cannot be
redistributed, so the frame the shipped model was fitted on is not in this repository. Costs
here are therefore higher than the one the README publishes. What is being compared is the
ordering, and an ordering survives a change of sample.


In [1]:
import pandas as pd

from credexp.utils import REPORTS_DIR

comparison = pd.read_csv(REPORTS_DIR / "model_comparison.csv")
comparison[["model", "variant", "n_applicants", "n_splits", "note"]]

,model,variant,n_applicants,n_splits,note
0,dummy,most_frequent,30751,5,"refuses nobody: the majority class, every time"
1,dummy,stratified,30751,5,a coin weighted by the base rate
2,lr,NaN,30751,5,"logistic regression, balanced class weights, s..."
3,lgbm,NaN,30751,5,"gradient-boosted trees, balanced class weights..."


## What each family costs per applicant

The cost is the business one: a missed default is worth ten wrongful refusals, and the
threshold of each family is optimised on one fold and applied to the next, so no fold both
chooses and scores its own operating point.


In [2]:
ranked = comparison.sort_values("business_cost_per_row")
ranked[["model", "variant", "business_cost_per_row", "roc_auc_mean", "pr_auc_mean"]]

,model,variant,business_cost_per_row,roc_auc_mean,pr_auc_mean
3,lgbm,NaN,0.5472,0.7433,0.2324
2,lr,NaN,0.5623,0.7348,0.2238
1,dummy,stratified,0.8050,0.5071,0.0821
0,dummy,most_frequent,0.8075,0.5000,0.0807


## The two distances that matter

Between the baselines and anything that learns, and between the two things that learn.


In [3]:
baseline = comparison.loc[comparison["model"] == "dummy", "business_cost_per_row"].min()
regression = comparison.loc[comparison["model"] == "lr", "business_cost_per_row"].iloc[0]
trees = comparison.loc[comparison["model"] == "lgbm", "business_cost_per_row"].iloc[0]

pd.Series(
    {
        "best trivial baseline": round(baseline, 4),
        "logistic regression": round(regression, 4),
        "gradient-boosted trees": round(trees, 4),
        "learning at all, vs the baseline": round(baseline - regression, 4),
        "trees over the regression": round(regression - trees, 4),
    }
)

best trivial baseline               0.8050
logistic regression                 0.5623
gradient-boosted trees              0.5472
learning at all, vs the baseline    0.2427
trees over the regression           0.0151
dtype: float64

## What this says

Learning is worth about a quarter of the cost per applicant. Choosing trees over a logistic
regression is worth an order of magnitude less than that, on a feature set built to make the
regression competitive: 796 columns, most of them aggregates a tree would have had to find
for itself.

That is the honest shape of the result. The family was still chosen on it, because the gap
is consistent across folds and costs nothing to keep, but a report that stopped at « trees
win » would leave a reader thinking the choice carried the project.

The threshold is what carries the project, and `docs/protocol.md` says how it is chosen.


In [4]:
spread = comparison[["model", "variant", "roc_auc_std", "pr_auc_std"]].copy()
spread["cost"] = comparison["business_cost_per_row"]
spread["folds"] = comparison["n_splits"]
spread

,model,variant,roc_auc_std,pr_auc_std,cost,folds
0,dummy,most_frequent,0.0000,0.0001,0.8075,5
1,dummy,stratified,0.0054,0.0011,0.8050,5
2,lr,NaN,0.0118,0.0158,0.5623,5
3,lgbm,NaN,0.0105,0.0080,0.5472,5
